In [ ]:
# ===================== SHARED SETUP: copy this whole block into the first cell, unchanged =====================
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

FAMILY = "random_forest"          # your model family (already filled in in your task file)
SEED = 42
# The repository root, found from wherever the notebook runs (Jupyter starts in the notebook's own folder).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "Phase 2").exists())
# The data file: in the repository root (where you save it), or in Phase 2/reports/ (the lead's copy).
DATA = next((p for p in (ROOT / "pre_training_data.xlsx", ROOT / "Phase 2" / "reports" / "pre_training_data.xlsx")
             if p.exists()), None)
assert DATA, f"pre_training_data.xlsx not found: save it in the repository root ({ROOT})"
OUT = ROOT / "Phase 2" / "notebooks" / FAMILY                    # your folder: notebook + committed, synthetic-only outputs
PRIVATE = ROOT / "Phase 2" / "reports" / "tasks" / FAMILY       # NOT committed (gitignored): anything derived from real rows
OUT.mkdir(parents=True, exist_ok=True); PRIVATE.mkdir(parents=True, exist_ok=True)

# ---------- data ----------
m = pd.read_excel(DATA)
BOOLS = ["label_usable", "post_call_row", "paid_30d", "paid_after_call_30d", "inbound", "spoke_with_agent",
         "collection_attempted", "promised_to_pay", "non_committal", "contested_or_refused", "pursued_insurance",
         "transferred", "action_shared_balance", "action_payment_plan", "action_financial_assistance",
         "action_statement_or_bill", "action_insurance", "asked_financial_assistance"]
for c in BOOLS:
    m[c] = m[c].astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0}).astype(float)
u = m[m.label_usable == 1]
SYN_TRAIN = u[(u.data_type == "synthetic") & (u.split == "train")]
SYN_TEST = u[(u.data_type == "synthetic") & (u.split == "test")]
REAL = u[u.data_type == "real"]
TARGET = {"pre": "paid_30d", "post": "paid_after_call_30d"}

def rows(df, target):
    """Pre-call model: every usable row. Post-call model: only calls without a payment on the call."""
    return df[df.post_call_row == 1] if target == "post" else df

# ---------- features ----------
PRE_ALL = ["n_accounts", "n_hospital_accounts", "n_physician_accounts", "placement_balance", "balance_at_t",
           "days_since_first_placement", "days_since_last_placement", "days_since_last_discharge", "n_payments_90d",
           "amount_paid_90d", "n_payments_365d", "amount_paid_365d", "n_payments_ever", "amount_paid_ever",
           "days_since_last_payment", "n_insurance_payments_365d", "amount_insurance_365d", "n_adjustments_365d",
           "amount_adjustments_365d", "paid_share_so_far", "last_payment_amount"]
CALL = ["inbound", "spoke_with_agent", "collection_attempted", "promised_to_pay", "non_committal", "contested_or_refused",
        "pursued_insurance", "transferred", "action_shared_balance", "action_payment_plan", "action_financial_assistance",
        "action_statement_or_bill", "action_insurance", "n_actions", "asked_financial_assistance", "talk_seconds",
        "caller_sentiment_overall", "caller_sentiment_end", "caller_sentiment_shift", "agent_sentiment_overall", "qa_score"]
CURRENT_PRE = ["n_accounts", "n_hospital_accounts", "n_physician_accounts", "balance_at_t", "days_since_first_placement",
               "days_since_last_placement", "n_payments_90d", "n_payments_365d", "days_since_last_payment",
               "n_insurance_payments_365d", "n_adjustments_365d"]
TIER12_PRE = ["n_payments_90d", "days_since_last_payment", "n_payments_365d", "paid_share_so_far", "last_payment_amount",
              "days_since_first_placement", "days_since_last_placement"]
SETS = {
    "pre": {
        "stable": ["n_payments_90d", "days_since_last_payment"],
        "top5": ["n_payments_90d", "days_since_last_payment", "n_payments_365d", "amount_adjustments_365d",
                 "days_since_first_placement"],
        "l1_one_se": ["n_payments_90d", "days_since_last_payment", "n_payments_365d", "amount_adjustments_365d",
                      "days_since_first_placement", "paid_share_so_far", "n_physician_accounts"],
        "tiers12": TIER12_PRE,
        "current": CURRENT_PRE,
        "all": PRE_ALL,
    },
    "post": {
        "stable": ["n_payments_90d", "days_since_last_payment", "action_payment_plan", "promised_to_pay",
                   "days_since_first_placement"],
        "top5": ["n_payments_90d", "days_since_last_payment", "action_payment_plan", "promised_to_pay",
                 "days_since_last_placement"],
        "l1_one_se": ["n_payments_90d", "days_since_last_payment", "action_payment_plan", "promised_to_pay",
                      "days_since_last_placement", "days_since_first_placement", "paid_share_so_far",
                      "amount_adjustments_365d", "spoke_with_agent", "n_payments_365d", "action_shared_balance",
                      "action_financial_assistance", "n_physician_accounts", "action_statement_or_bill"],
        "tiers12": TIER12_PRE + ["action_payment_plan", "promised_to_pay", "spoke_with_agent"],
        "current": CURRENT_PRE + CALL,
        "all": PRE_ALL + CALL,
    },
}
NEVER_USE = {"paid_30d", "paid_after_call_30d", "paid_amount_30d", "days_to_first_payment", "paid_fraction_30d",
             "paid_on_call", "payment_on_call", "label_usable", "label_reliable", "window_complete", "data_type",
             "split", "post_call_row", "sentiment_source", "qa_source", "guarantor_id", "call_id", "snapshot_id",
             "as_of", "agent", "tcn_record_id", "source", "ever_paid", "n_prior_contacts_90d", "n_prior_calls",
             "days_since_last_contact"}
assert not any(set(f) & NEVER_USE for s in SETS.values() for f in s.values()), "a forbidden column is in a feature set"
for c in PRE_ALL + CALL:
    m[c] = pd.to_numeric(m[c], errors="coerce")
SYN_TRAIN, SYN_TEST, REAL = (d.assign(**{c: pd.to_numeric(d[c], errors="coerce") for c in PRE_ALL + CALL})
                             for d in (SYN_TRAIN, SYN_TEST, REAL))

# ---------- preprocessing (put it INSIDE your pipeline, so it is fitted on training rows only) ----------
SKEWED = ["n_accounts", "n_hospital_accounts", "n_physician_accounts", "placement_balance", "balance_at_t",
          "days_since_first_placement", "days_since_last_placement", "days_since_last_discharge", "n_payments_90d",
          "amount_paid_90d", "n_payments_365d", "amount_paid_365d", "n_payments_ever", "amount_paid_ever",
          "days_since_last_payment", "n_insurance_payments_365d", "amount_insurance_365d", "n_adjustments_365d",
          "amount_adjustments_365d", "last_payment_amount", "n_actions", "talk_seconds"]

class Prep(BaseEstimator, TransformerMixin):
    """days_since_last_payment is empty when the patient never paid: fill with (train max + 1) and add a 0/1
    'never_paid' column. Any other gap -> 0. Optional log1p on skewed non-negative columns."""
    def __init__(self, log=True):
        self.log = log
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.has_dslp_ = "days_since_last_payment" in X
        self.fill_ = (np.nanmax(X["days_since_last_payment"].to_numpy()) + 1) if self.has_dslp_ else None
        self.cols_ = list(X.columns) + (["never_paid"] if self.has_dslp_ else [])
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        if self.has_dslp_:
            X["never_paid"] = X["days_since_last_payment"].isna().astype(float)
            X["days_since_last_payment"] = X["days_since_last_payment"].fillna(self.fill_)
        X = X.fillna(0.0)
        if self.log:
            for c in X.columns:
                if c in SKEWED:
                    X[c] = np.log1p(X[c].clip(lower=0))
        return X[self.cols_]
    def get_feature_names_out(self, input_features=None):
        return np.array(self.cols_)

# ---------- folds: everyone uses exactly these ----------
def folds(df, target, k=5, seed=SEED):
    """Grouped by patient: all calls of one patient are in the same fold."""
    y = df[TARGET[target]].astype(int)
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(df, y, df["guarantor_id"]))

# ---------- metrics ----------
def calibration(y, p):
    """Slope and intercept of the logistic recalibration (ideal: 1 and 0)."""
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(z, y)
    return float(lr.coef_[0][0]), float(lr.intercept_[0])

def metrics(y, p):
    y, p = np.asarray(y).astype(int), np.asarray(p, float)
    top = np.argsort(-p)[: max(1, int(round(0.2 * len(p))))]
    slope, icpt = calibration(y, p)
    return {"auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p), "brier": brier_score_loss(y, p),
            "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "calib_slope": slope, "calib_intercept": icpt,
            "top20_capture": y[top].sum() / max(y.sum(), 1), "mean_pred": p.mean(), "pay_rate": y.mean()}

def boot_ci(y, p, n=2000, seed=SEED):
    rng = np.random.default_rng(seed); y, p = np.asarray(y).astype(int), np.asarray(p, float); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            v.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def fit(model, X, y, w=None):
    """Fit with sample weights when the final step supports them (pipelines: last step must be named 'model')."""
    if w is None:
        return model.fit(X, y)
    try:
        return model.fit(X, y, model__sample_weight=w)
    except (TypeError, ValueError):
        return model.fit(X, y)          # estimator without sample_weight: real rows count once each

# ---------- 1) synthetic cross-validation: the main way to compare and choose ----------
def cv_eval(make_model, feats, target):
    """Grouped 5-fold CV on the synthetic training rows. Returns mean metrics, their SE, and out-of-fold predictions."""
    df = rows(SYN_TRAIN, target); y = df[TARGET[target]].astype(int).to_numpy()
    oof = np.full(len(df), np.nan); per = []; t0 = time.time()
    for tr, te in folds(df, target):
        mdl = fit(make_model(), df.iloc[tr][feats], y[tr])
        oof[te] = mdl.predict_proba(df.iloc[te][feats])[:, 1]
        per.append(metrics(y[te], oof[te]))
    per = pd.DataFrame(per)
    out = {f"cv_{k}": v for k, v in per.mean().items()}
    out["cv_auc_se"] = per["auc"].std(ddof=1) / np.sqrt(len(per))
    out["fit_seconds"] = round(time.time() - t0, 1)
    return out, pd.DataFrame({"call_id": df["call_id"].to_numpy(), "y": y, "p": oof})

# ---------- 2) real rows, measured fairly: repeated grouped cross-validation over the real patients ----------
REAL_LOOKS = {"n": 0}
def real_eval(make_model, feats, target, real_weight=0.0, repeats=10):
    """Score the real rows fairly. The real patients are split into 5 stratified groups (all calls of a patient
    together); each group is scored by a model trained on synthetic data + the OTHER real groups (each real row with
    weight real_weight; 0 = synthetic only). Repeated with 10 different splits. No real row is ever scored by a model
    that saw it.
    real_auc = AUC computed WITHIN each held-out group, averaged over the 50 groups. (Pooling the groups' predictions
    is not used for AUC: many real rows have identical features, and tiny differences between the groups' models
    would rank them at random. Leave-one-patient-out is not used either: it biases AUC downwards.)
    With real_weight = 0 there is one model, so the single-model AUC and its 95% bootstrap interval are also given.
    Every call counts as one 'look' at the real data."""
    REAL_LOOKS["n"] += 1
    syn = rows(SYN_TRAIN, target); real = rows(REAL, target).reset_index(drop=True)
    ys, yr = syn[TARGET[target]].astype(int).to_numpy(), real[TARGET[target]].astype(int).to_numpy()
    base = fit(make_model(), syn[feats], ys) if real_weight == 0 else None
    within, preds = [], []
    for r in range(repeats):
        p_r = np.full(len(real), np.nan)
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(real, yr, real["guarantor_id"]):
            if base is not None:
                mdl = base
            else:
                X = pd.concat([syn[feats], real.iloc[tr][feats]]); y = np.r_[ys, yr[tr]]
                w = np.r_[np.ones(len(ys)), np.full(len(tr), float(real_weight))]
                mdl = fit(make_model(), X, y, w)
            p_r[te] = mdl.predict_proba(real.iloc[te][feats])[:, 1]
            if yr[te].min() != yr[te].max():
                within.append(roc_auc_score(yr[te], p_r[te]))
        preds.append(p_r)
    p = np.mean(preds, axis=0)
    met = metrics(yr, p)
    out = {"real_auc": float(np.mean(within)), "real_auc_folds_sd": float(np.std(within, ddof=1))}
    out.update({f"real_{k}": v for k, v in met.items() if k != "auc"})
    if base is not None:   # one model for every real row: a plain AUC with a bootstrap interval is valid
        lo, hi = boot_ci(yr, p)
        out.update({"real_auc_single_model": met["auc"], "real_auc_lo": lo, "real_auc_hi": hi})
    out.update({"real_n": len(yr), "real_weight": real_weight, "real_looks_so_far": REAL_LOOKS["n"]})
    return out, pd.DataFrame({"call_id": real["call_id"].to_numpy(), "y": yr, "p": p})

# ---------- baseline everyone must beat: the rule "paid before, most recent first" ----------
def rule_scores(df):
    """Patients who paid before first, the most recent payers first. No training involved."""
    paid = df["days_since_last_payment"].notna().astype(float)
    return paid * 100000 - df["days_since_last_payment"].fillna(0)

def rule_real_auc(target, repeats=10):
    """The rule scored on the real rows exactly like real_eval (within-fold AUC, same splits): the number to beat."""
    real = rows(REAL, target).reset_index(drop=True); yr = real[TARGET[target]].astype(int).to_numpy()
    s = rule_scores(real).to_numpy(); within = []
    for r in range(repeats):
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(real, yr, real["guarantor_id"]):
            if yr[te].min() != yr[te].max():
                within.append(roc_auc_score(yr[te], s[te]))
    return float(np.mean(within))

# ---------- 3) final: fit once, score the synthetic TEST split once, save predictions ----------
def final_model(make_model, feats, target, real_weight=0.0, variant="", set_name="", params=None, notes=""):
    """Call this ONCE per target, for your chosen model. Fits on synthetic train (+ all real rows with real_weight),
    scores the synthetic test split, re-measures real fairly, logs a 'FINAL' row and saves predictions:
    synthetic out-of-fold + test predictions -> OUT (committed); real predictions -> PRIVATE (never committed)."""
    syn, test = rows(SYN_TRAIN, target), rows(SYN_TEST, target)
    ys, yt = syn[TARGET[target]].astype(int).to_numpy(), test[TARGET[target]].astype(int).to_numpy()
    if real_weight > 0:
        real = rows(REAL, target)
        X = pd.concat([syn[feats], real[feats]]); y = np.r_[ys, real[TARGET[target]].astype(int).to_numpy()]
        w = np.r_[np.ones(len(ys)), np.full(len(real), float(real_weight))]
        mdl = fit(make_model(), X, y, w)
    else:
        mdl = fit(make_model(), syn[feats], ys)
    p_test = mdl.predict_proba(test[feats])[:, 1]
    st = {f"test_{k}": v for k, v in metrics(yt, p_test).items()}
    st["test_auc_lo"], st["test_auc_hi"] = boot_ci(yt, p_test)
    st["test_rule_auc"] = roc_auc_score(yt, rule_scores(test))
    cv, oof = cv_eval(make_model, feats, target)
    real_m, real_p = real_eval(make_model, feats, target, real_weight)
    real_m["real_rule_auc"] = rule_real_auc(target)
    log_result("FINAL " + variant, target, set_name, feats, params or {}, cv, real_m, st, notes)
    pd.concat([oof.assign(kind="synthetic_oof"), pd.DataFrame({"call_id": test["call_id"].to_numpy(), "y": yt,
               "p": p_test, "kind": "synthetic_test"})]).assign(family=FAMILY, target=target, variant=variant,
               feature_set=set_name).to_csv(OUT / f"synthetic_predictions_{target}.csv", index=False)
    real_p.assign(family=FAMILY, target=target, variant=variant, feature_set=set_name).to_csv(
        PRIVATE / f"real_predictions_{target}.csv", index=False)
    return mdl, {**cv, **real_m, **st}

# ---------- logging: one row per experiment ----------
RESULTS = OUT / "results.csv"
def log_result(variant, target, set_name, feats, params, cv, real=None, synthetic_test=None, notes=""):
    row = {"family": FAMILY, "variant": variant, "target": target, "feature_set": set_name, "n_features": len(feats),
           "params": json.dumps(params, default=str), **cv, **(real or {}), **(synthetic_test or {}), "notes": notes,
           "logged_at": pd.Timestamp.now().isoformat(timespec="seconds")}
    df = pd.DataFrame([row])
    df.to_csv(RESULTS, mode="a", header=not RESULTS.exists(), index=False)
    return row

print(f"synthetic train: pre {len(rows(SYN_TRAIN, 'pre')):,} / post {len(rows(SYN_TRAIN, 'post')):,} rows; "
      f"synthetic test: pre {len(rows(SYN_TEST, 'pre')):,} / post {len(rows(SYN_TEST, 'post')):,}; "
      f"real: pre {len(rows(REAL, 'pre'))} / post {len(rows(REAL, 'post'))}")
# =========================================== END OF SHARED SETUP ===========================================

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

target = "pre"
set_name = "tiers12"
features = SETS[target][set_name]

params = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 20,
    "max_features": "sqrt",
    "random_state": SEED,
    "n_jobs": -1,
}

def make_forest():
    return Pipeline([
        ("prep", Prep(log=False)),
        ("model", RandomForestClassifier(**params)),
    ])

cv, _ = cv_eval(make_forest, features, target)
log_result("random forest", target, set_name, features, params, cv)

print(f"CV AUC: {cv['cv_auc']:.3f}")
print(f"CV AUC standard error: {cv['cv_auc_se']:.3f}")

In [ ]:
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline

# Save the result from the first pre-call Random Forest cell.
first_cv = cv.copy()

algorithms = {
    "random forest": RandomForestClassifier,
    "extra trees": ExtraTreesClassifier,
}

feature_sets = ["stable", "top5", "l1_one_se", "tiers12", "current", "all"]

base_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 20,
    "max_features": "sqrt",
    "random_state": SEED,
    "n_jobs": -1,
}

def make_pipeline(model_class, settings):
    return Pipeline([
        ("prep", Prep(log=False)),
        ("model", model_class(**settings)),
    ])

screening_rows = []

for target in ["pre", "post"]:
    for algorithm, model_class in algorithms.items():
        for set_name in feature_sets:
            feats = SETS[target][set_name]

            if (target, algorithm, set_name) == ("pre", "random forest", "tiers12"):
                scores = first_cv
            else:
                def make_model():
                    return make_pipeline(model_class, base_params)

                scores, _ = cv_eval(make_model, feats, target)
                log_result(algorithm, target, set_name, feats, base_params, scores)

            screening_rows.append({
                "target": target,
                "algorithm": algorithm,
                "feature_set": set_name,
                "n_features": len(feats),
                **scores,
            })
            print(target, algorithm, set_name, f"AUC={scores['cv_auc']:.3f}")

screening = pd.DataFrame(screening_rows)
columns = [
    "target", "algorithm", "feature_set", "n_features",
    "cv_auc", "cv_auc_se", "cv_brier", "cv_calib_slope",
]

display(
    screening.sort_values(["target", "cv_auc"], ascending=[True, False])[columns]
)

In [ ]:
ranked_sets = (
    screening
    .sort_values("cv_auc", ascending=False)
    .drop_duplicates(subset=["target", "feature_set"])
    .groupby("target", sort=False)
    .head(3)
)

display(ranked_sets[columns])

In [ ]:
from sklearn.model_selection import ParameterSampler

search_space = {
    "n_estimators": [300, 600],
    "max_depth": [None, 8, 16],
    "min_samples_leaf": [1, 5, 20, 50, 100],
    "max_features": ["sqrt", 0.3, 0.5, 0.8],
    "class_weight": [None, "balanced_subsample"],
}

starter_settings = {**base_params, "class_weight": None}
sampled_settings = [
    settings
    for settings in ParameterSampler(search_space, n_iter=16, random_state=SEED)
    if {**base_params, **settings} != starter_settings
][:15]

tuning_rows = []

for target, set_name in ranked_sets[["target", "feature_set"]].itertuples(
    index=False, name=None
):
    features = SETS[target][set_name]

    for algorithm, model_class in algorithms.items():
        for settings in sampled_settings:
            params = {**base_params, **settings}

            def make_model(model_class=model_class, params=params):
                return make_pipeline(model_class, params)

            scores, _ = cv_eval(make_model, features, target)
            log_result(algorithm, target, set_name, features, params, scores)

            tuning_rows.append({
                "target": target,
                "algorithm": algorithm,
                "feature_set": set_name,
                "n_features": len(features),
                "params": params,
                **scores,
            })

        print(f"Finished {target} | {set_name} | {algorithm}", flush=True)

tuning = pd.DataFrame(tuning_rows)

for target in ["pre", "post"]:
    print(f"\nBest tuned runs for {target}")
    display(
        tuning.loc[tuning["target"] == target]
        .sort_values("cv_auc", ascending=False)
        .head(5)[columns + ["params"]]
    )

In [ ]:
#Inspecting full settings and choosing shortlist
display(
    tuning.loc[[138, 139, 110, 27, 35, 5],
               ["target", "algorithm", "feature_set", "params"]]
    .style.set_properties(**{"text-align": "left", "white-space": "normal"})
)

print("Full settings for pre-call candidate 139:")
print(tuning.loc[139, "params"])

In [ ]:
#Checking full settings for shortlisted candidates
candidate = tuning.loc[138]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

real_scores, _ = real_eval(make_candidate, features, target)
print(real_scores)

In [ ]:
candidate = tuning.loc[139]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

real_scores, _ = real_eval(make_candidate, features, target)
print(real_scores)

In [ ]:
candidate = tuning.loc[110]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

real_scores, _ = real_eval(make_candidate, features, target)
print(real_scores)

In [ ]:
candidate = tuning.loc[27]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

real_scores, _ = real_eval(make_candidate, features, target)
print(real_scores)

In [ ]:
candidate = tuning.loc[35]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

real_scores, _ = real_eval(make_candidate, features, target)
print(real_scores)

In [ ]:
candidate = tuning.loc[5]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

real_scores, _ = real_eval(make_candidate, features, target)
print(real_scores)

In [ ]:
candidate = tuning.loc[138]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

for weight in [1, 5, 20]:
    real_scores, _ = real_eval(
        make_candidate, features, target, real_weight=weight
    )
    print(
        f"Row {candidate.name} | {target} | weight={weight} | "
        f"real AUC={real_scores['real_auc']:.4f} | "
        f"top-20% capture={real_scores['real_top20_capture']:.1%}",
        flush=True,
    )

In [ ]:
candidate = tuning.loc[139]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

for weight in [1, 5, 20]:
    real_scores, _ = real_eval(
        make_candidate, features, target, real_weight=weight
    )
    print(
        f"Row {candidate.name} | {target} | weight={weight} | "
        f"real AUC={real_scores['real_auc']:.4f} | "
        f"top-20% capture={real_scores['real_top20_capture']:.1%}",
        flush=True,
    )

In [ ]:
candidate = tuning.loc[110]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

for weight in [1, 5, 20]:
    real_scores, _ = real_eval(
        make_candidate, features, target, real_weight=weight
    )
    print(
        f"Row {candidate.name} | {target} | weight={weight} | "
        f"real AUC={real_scores['real_auc']:.4f} | "
        f"top-20% capture={real_scores['real_top20_capture']:.1%}",
        flush=True,
    )

In [ ]:
candidate = tuning.loc[27]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

for weight in [1, 5, 20]:
    real_scores, _ = real_eval(
        make_candidate, features, target, real_weight=weight
    )
    print(
        f"Row {candidate.name} | {target} | weight={weight} | "
        f"real AUC={real_scores['real_auc']:.4f} | "
        f"top-20% capture={real_scores['real_top20_capture']:.1%}",
        flush=True,
    )

In [ ]:
candidate = tuning.loc[35]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

for weight in [1, 5, 20]:
    real_scores, _ = real_eval(
        make_candidate, features, target, real_weight=weight
    )
    print(
        f"Row {candidate.name} | {target} | weight={weight} | "
        f"real AUC={real_scores['real_auc']:.4f} | "
        f"top-20% capture={real_scores['real_top20_capture']:.1%}",
        flush=True,
    )

In [ ]:
candidate = tuning.loc[5]
target = candidate['target']
features = SETS[target][candidate['feature_set']]
model_class = algorithms[candidate['algorithm']]
settings = candidate['params']

def make_candidate():
    return make_pipeline(model_class, settings)

for weight in [1, 5, 20]:
    real_scores, _ = real_eval(
        make_candidate, features, target, real_weight=weight
    )
    print(
        f"Row {candidate.name} | {target} | weight={weight} | "
        f"real AUC={real_scores['real_auc']:.4f} | "
        f"top-20% capture={real_scores['real_top20_capture']:.1%}",
        flush=True,
    )

## Final Model Selection

Based off all real AUC results and chosen weights, here are what I beleive are the current best choices:

| Target | Selected tuning row | Algorithim | Feature set | Real Weight | Synthetic CV AUC | Eligibility Cutoff | Real AUC | 
|---|---:|---|---|---:|---:|---:|---:|
| Pre call | 110 | Extra Trees | tiers12 | 5 | 0.712557 | 0.705462 | 0.8414 | 
| Post call | 5 | Random Forest | tiers12 | 1 | 0.802826 | 0.796814 | 0.7469 | 

Both models cleared their synthetic CV cutoffs, and their real AUCS were within 0.02 of the respective leaders, so I selected the configurations with fewer input featuress: 7 instead of 21 for pre call, and 10 instead of 14 for post call

In [ ]:
import json
import pandas as pd

if RESULTS.exists():
    pd.read_csv(RESULTS)  # Check that the existing results file can be read.

def log_result(variant, target, set_name, feats, params, cv,
               real=None, synthetic_test=None, notes=""):
    row = {
        "family": FAMILY,
        "variant": variant,
        "target": target,
        "feature_set": set_name,
        "n_features": len(feats),
        "params": json.dumps(params, default=str),
        **cv,
        **(real or {}),
        **(synthetic_test or {}),
        "notes": notes,
        "logged_at": pd.Timestamp.now().isoformat(timespec="seconds"),
    }

    existing = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()
    updated = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    temporary = RESULTS.with_suffix(".tmp")
    updated.to_csv(temporary, index=False)
    temporary.replace(RESULTS)
    return row

print("Logger ready.")

In [ ]:
final_results = {}

for row_id, weight in [(110, 5), (5, 1)]:
    candidate = tuning.loc[row_id]
    target = candidate["target"]
    algorithm = candidate["algorithm"]
    set_name = candidate["feature_set"]
    features = SETS[target][set_name]
    params = candidate["params"]

    def make_candidate(model_class=algorithms[algorithm], settings=params):
        return make_pipeline(model_class, settings)

    _, scores = final_model(
        make_candidate,
        features,
        target,
        real_weight=weight,
        variant=algorithm,
        set_name=set_name,
        params=params,
        notes=f"Selected tuning row {row_id}, real weight {weight}",
    )

    final_results[target] = scores
    print(f"Finished {target}: row {row_id}, weight {weight}", flush=True)

final_table = pd.DataFrame.from_dict(final_results, orient="index")
display(final_table[[
    "real_auc", "real_rule_auc",
    "test_auc", "test_rule_auc",
    "test_auc_lo", "test_auc_hi",
    "test_calib_slope", "test_top20_capture",
]])

In [ ]:
import sys
import numpy as np
import pandas as pd
from importlib.metadata import version
from sklearn.inspection import permutation_importance

for row_id, real_weight in [(110, 5), (5, 1)]:
    candidate = tuning.loc[row_id]
    target = candidate["target"]
    features = SETS[target][candidate["feature_set"]]
    target_column = TARGET[target]

    synthetic = rows(SYN_TRAIN, target)
    real_training = rows(REAL, target)
    model_class = algorithms[candidate["algorithm"]]
    fold_importance = []

    for train_idx, val_idx in folds(synthetic, target):
        train = synthetic.iloc[train_idx]
        validation = synthetic.iloc[val_idx]

        X_train = pd.concat([train[features], real_training[features]])
        y_train = pd.concat([train[target_column], real_training[target_column]]).astype(int)
        weights = np.r_[np.ones(len(train)), np.full(len(real_training), real_weight)]

        model = make_pipeline(model_class, candidate["params"])
        model.fit(X_train, y_train, model__sample_weight=weights)

        importance = permutation_importance(
            model,
            validation[features],
            validation[target_column].astype(int),
            scoring="roc_auc",
            n_repeats=5,
            random_state=SEED,
        )
        fold_importance.append(importance.importances_mean)

    summary = pd.Series(
        np.mean(fold_importance, axis=0),
        index=features,
        name="Mean validation AUC drop",
    ).sort_values(ascending=False)

    print(f"\n{target}: row {row_id}, real weight {real_weight}")
    display(summary.head(5).round(4))

print("\nPython:", sys.version.split()[0])
for package in ["numpy", "pandas", "scikit-learn", "openpyxl", "ipykernel", "nbstripout"]:
    print(f"{package}: {version(package)}")

In [ ]:
import hashlib, pandas as pd
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())   # works from any folder
m = pd.read_excel(ROOT / "pre_training_data.xlsx")
key = m.sort_values("call_id")[["call_id", "split", "paid_30d", "paid_after_call_30d"]].astype(str).agg("|".join, axis=1)
print(m.shape, hashlib.sha256("\n".join(key).encode()).hexdigest()[:16])
print(m.groupby(["data_type", "split"]).agg(rows=("call_id", "size"), usable=("label_usable", "sum")))